In [1]:
# Imports

import os
from dotenv import load_dotenv
import requests
import json
import pandas as pd

In [2]:
load_dotenv()

api_key = os.getenv("API_KEY")
repo_owner = 'bedbad'
repo_name = 'justpyplot'

url = "https://api.github.com/graphql"
headers = {
    "Authorization": f"Bearer {api_key}"
}

def get_folder_contents(expression="HEAD:"):
    """Fetches the contents of a specific folder path."""
    query = """
    query($owner: String!, $name: String!, $expression: String!) {
      repository(owner: $owner, name: $name) {
        object(expression: $expression) {
          ... on Tree {
            entries {
              name
              path
              type
              extension
            }
          }
        }
      }
    }
    """
    variables = {
        "owner": repo_owner, 
        "name": repo_name, 
        "expression": expression
    }
    response = requests.post(url, json={'query': query, 'variables': variables}, headers=headers)
    return response.json()

all_items = []
# Start at the root of the repository
folders_to_check = ["HEAD:"]

print("Mapping directory structure... this may take a moment for large repos.")

while folders_to_check:
    current_folder = folders_to_check.pop(0)
    data = get_folder_contents(current_folder)
    
    # Safely extract entries
    try:
        entries = data['data']['repository']['object']['entries']
    except (KeyError, TypeError):
        continue
        
    for entry in entries:
        all_items.append({
            'Name': entry['name'],
            'Path': entry['path'],
            'Type': entry['type'], # 'blob' for files, 'tree' for folders
            'Extension': entry.get('extension', '')
        })
        
        # If the item is a folder (tree), queue it up to fetch its contents next
        if entry['type'] == 'tree':
            folders_to_check.append(f"HEAD:{entry['path']}/")

# Dump a sample to json for reference
with open("directory_sample.json", "w") as file:
    json.dump(all_items[:100], file, indent=4)
    
print(f"Done! Found {len(all_items)} total files and folders.")

Mapping directory structure... this may take a moment for large repos.


Done! Found 38 total files and folders.


In [3]:
df = pd.DataFrame(all_items)

# Add a boolean column for easier filtering later
df['Is_Folder'] = df['Type'] == 'tree'

# Sort alphabetically by path so it looks like a proper directory tree
df_sorted = df.sort_values(by='Path').reset_index(drop=True)

# Let's take a look at just the files (blobs)
df_files_only = df_sorted[df_sorted['Type'] == 'blob']

df_sorted.head(10)

,Name,Path,Type,Extension,Is_Folder
0,.github,.github,tree,,True
1,funding.yaml,.github/funding.yaml,blob,.yaml,False
2,workflows,.github/workflows,tree,,True
3,python-publish.yml,.github/workflows/python-publish.yml,blob,.yml,False
4,.gitignore,.gitignore,blob,,False
5,.readthedocs.yaml,.readthedocs.yaml,blob,.yaml,False
6,CONTRIBUTING.md,CONTRIBUTING.md,blob,.md,False
7,LICENSE,LICENSE,blob,,False
8,README.md,README.md,blob,.md,False
9,docs,docs,tree,,True


Create CSV File

In [4]:
# Select the most useful columns for your database
final_df = df_sorted[['Path', 'Name', 'Extension', 'Type', 'Is_Folder']]

final_df.to_csv("repo_directory_structure.csv", index=False)
print("Saved directory structure to CSV!")

Saved directory structure to CSV!
